# V4: F1 + 엄격한 뒤집기 (어려운 한글 행 → v20, 단 112·HanSunwook·Erde 셋이 같은 다른 답이면 112)
**GPU 런타임(L4)**으로 위에서부터 실행하세요. 마지막에 `V4: PASS`가 나오면 `out/V4.zip`이 생깁니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys
D = "/content/drive/MyDrive/Projects/항만문제해결을위한AI기술공모전/combo_kit"
os.chdir(D)
!ls -la build_vote4.py build_fuse.py er_submit.zip v20.zip 112_submit.zip 1031_08_hyb_w.zip
!pip -q install lightgbm==4.6.0 catboost==1.2.8 rapidfuzz==3.14.3 transformers==4.57.6 accelerate==1.9.0 scikit-learn==1.5.2
W = "/content/combo"; os.makedirs(W, exist_ok=True); os.makedirs(f"{D}/out", exist_ok=True)
for f in ("V4.zip",):
    if os.path.exists(f"{W}/{f}"): os.remove(f"{W}/{f}")


In [ ]:
# 조립 (약 15분)
!python build_vote4.py {W}/V4.zip v20.zip 1031_08_hyb_w.zip er_submit.zip 112_submit.zip 4


In [ ]:
# 점검 (약 20분): 3,000행 + 영문 불변 + 600행 행 독립성. 통과할 때만 out/으로 복사
zp = f"{W}/V4.zip"
c = subprocess.run([sys.executable, "check_combo.py", zp, f"{W}/work_v4"], capture_output=True, text=True)
log = c.stdout + "\n" + c.stderr
open(f"{W}/V4_check.log", "w").write(log)
print("\n".join(l for l in log.splitlines() if l.startswith(("[1]", "[2]", "[3]", "zip size", "OK", "PROBLEM", "WARNING ko", "WARNING hs", "WARNING er")) or "predictions to output" in l))
bad = [k for k in ("WARNING ko", "WARNING hs", "WARNING er") if k in log]
last = (c.stdout.strip().splitlines() or [""])[-1]
if c.returncode == 0 and last == "OK" and not bad:
    shutil.copy(zp, f"{D}/out/V4.zip"); print("V4: PASS ->", round(os.path.getsize(f"{D}/out/V4.zip")/1e9, 2), "GB")
else:
    print("V4: FAIL - 올리지 마세요.", bad or last)
